# Phase 1: High-Fidelity TTS Generation (XTTS-v2 & Bark)

This notebook generates deepfake spoof speech samples using state-of-the-art autoregressive neural TTS engines:
1. **Coqui XTTS-v2** (Zero-shot multi-speaker voice cloning)
2. **Suno Bark** (Transformer-based text-to-audio with expressive non-speech artifacts)

### Instructions:
- Enable GPU accelerator: `Runtime` -> `Change runtime type` -> `T4 GPU`
- Run all cells sequentially. Output zip will be saved to your Google Drive or direct download.

In [ ]:
# 1. Environment & GPU Verification
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Please switch runtime to T4 GPU.")

In [ ]:
# 2. Install Dependencies
!pip install -q TTS transformers soundfile librosa

In [ ]:
# 3. Generation Setup
import os
from pathlib import Path
import pandas as pd
import soundfile as sf

OUT_DIR = Path("generated_tts_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS = [
    "Your account security verification code is four eight two nine one.",
    "Artificial intelligence speech models require robust testing across accents.",
    "The quick brown fox jumps gracefully over the lazy golden retriever.",
    "We noticed an unauthorized login attempt from a new mobile device.",
    "Good morning, thank you for calling customer service, how can I assist you today?",
    "Please confirm your registered phone number before we process the wire transfer.",
    "Heavy rainfall is expected across coastal districts throughout the weekend.",
    "Our team has completed the annual software security and compliance audit.",
    "Could you please send me the updated quarterly budget spreadsheet by tonight?",
    "Voice authentication systems protect sensitive personal identification records."
]

manifest_rows = []

In [ ]:
# 4. Synthesize with Coqui XTTS-v2
from TTS.api import TTS

print("Loading XTTS-v2 model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

xtts_dir = OUT_DIR / "xtts"
xtts_dir.mkdir(exist_ok=True)

for idx, prompt in enumerate(PROMPTS):
    out_file = xtts_dir / f"xtts_{idx:02d}.wav"
    print(f"Synthesizing XTTS clip {idx+1}/{len(PROMPTS)}...")
    tts_xtts.tts_to_file(
        text=prompt,
        speaker_wav="https://github.com/coqui-ai/TTS/raw/dev/tests/data/ljspeech/wavs/LJ001-0001.wav",
        language="en",
        file_path=str(out_file)
    )
    manifest_rows.append({
        "filename": out_file.name,
        "generator": "xtts_v2",
        "speaker": "xtts_cloned",
        "text": prompt,
        "language": "en",
        "label": "spoof",
        "split": "dev"
    })
print("XTTS-v2 generation complete.")

In [ ]:
# 5. Package Outputs & Manifest for Local Import
df = pd.DataFrame(manifest_rows)
manifest_path = OUT_DIR / "metadata.csv"
df.to_csv(manifest_path, index=False)

!zip -r colab_tts_v2.zip generated_tts_v2/
print("Packaged colab_tts_v2.zip successfully. Download and extract to data_v2/tts/")